# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_04 — Walk-Forward Optimization

### Research question

How do we optimise strategy parameters without using future information, then validate whether the selected parameters survive out of sample after costs, turnover, and regime changes?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
```

The previous notebooks built backtest and cost engines. This notebook focuses on **walk-forward optimisation**, the research process that decides parameters using only past data.

It covers:

1. parameter overfitting problem;
2. rolling and anchored walk-forward windows;
3. train/validation/test separation;
4. parameter grid search;
5. cost-aware objective functions;
6. selecting parameters only on validation windows;
7. applying chosen parameters to the next test window;
8. expanding versus rolling calibration;
9. robustness scorecards;
10. parameter stability;
11. performance degradation from validation to test;
12. turnover and cost-aware optimisation;
13. benchmark comparisons;
14. heatmaps by parameter pair;
15. regime-conditioned performance;
16. walk-forward equity curve;
17. probability of backtest overfit intuition;
18. governance flags;
19. audit checks;
20. saved outputs and manifest.

The key idea:

> Walk-forward optimisation does not prove a strategy works. It proves you at least gave the strategy a fair chance to fail before believing it.

## 1. Why walk-forward optimisation?

The bad research workflow:

```text
1. Try many parameters on the full dataset.
2. Pick the best one.
3. Report the backtest.
```

This is overfitting.

A better workflow:

```text
1. Train on past data.
2. Select parameters on a validation period.
3. Trade those parameters on a later unseen test period.
4. Roll forward and repeat.
```

At every point in time, the strategy only uses information that would have been available then.

## 2. Window structure

A walk-forward split can be:

### Rolling

```text
train:      [t0, t1]
validation:    [t1, t2]
test:               [t2, t3]
then roll forward
```

The training window has fixed length.

### Anchored / expanding

```text
train:      [start, t1]
validation:        [t1, t2]
test:                   [t2, t3]
then expand train
```

The training window grows through time.

This notebook implements both.

## 3. Parameter selection

For each candidate parameter set $\theta$, evaluate validation performance:

$$
\begin{aligned}
Score(\theta) &= Sharpe_{val} \\
&\quad - penalty_{turnover} \\
&\quad - penalty_{drawdown}
\end{aligned}
$$

Then choose:

$$
\theta^* = \arg\max_\theta Score(\theta)
$$

The selected $\theta^*$ is then applied to the **test** window.

The test window must not influence selection.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class WalkForwardConfig:
    n_days: int = 2_200
    annualisation: int = 252
    seed: int = 42
    train_window: int = 504
    validation_window: int = 126
    test_window: int = 63
    step_size: int = 63
    anchored_train: bool = False
    gross_target: float = 1.20
    max_abs_weight: float = 0.25
    transaction_cost_bps: float = 4.0
    borrow_cost_ann: float = 0.015
    financing_cost_ann: float = 0.035
    cvar_alpha: float = 0.95
    turnover_penalty: float = 0.25
    drawdown_penalty: float = 0.50
    min_validation_days: int = 60

config = WalkForwardConfig()
config

## 5. Simulate a multi-asset research dataset

We create synthetic daily returns with:

- multiple asset classes;
- regimes;
- stress episodes;
- weak momentum / reversal predictability;
- changing volatility.

The goal is not to create magical alpha. The goal is to test whether a parameter-selection process is honest.

In [ ]:
def simulate_walk_forward_data(config: WalkForwardConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND", "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.006, 0.007, 0.035])
    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.15,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.15,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])
    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    ann_mu = pd.Series({
        "US_EQ": 0.070,
        "EU_EQ": 0.060,
        "EM_EQ": 0.080,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND": 0.08,
        "REIT": 0.11,
        "BTC_PROXY": 0.52,
    })

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_multiplier**2)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=6, size=len(assets)) * np.sqrt((6 - 2) / 6)
        eps = eps * idio_vol_ann.values / np.sqrt(config.annualisation) * vol_multiplier * 0.35

        # Weak regime-dependent predictability.
        predictive_tilt = np.zeros(len(assets))
        if t > 20:
            recent_market = asset_returns[max(0, t-20):t, assets.index("US_EQ")].mean()
            if recent_market > 0:
                predictive_tilt[assets.index("US_EQ")] += 0.00015
                predictive_tilt[assets.index("EU_EQ")] += 0.00010
            if regime[t - 1] == 1:
                predictive_tilt[assets.index("TREND")] += 0.00055
                predictive_tilt[assets.index("FX_CARRY")] -= 0.00035

        asset_returns[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps + predictive_tilt
        factor_returns[t] = f

    returns = pd.DataFrame(asset_returns, index=dates, columns=assets)
    prices = 100 * np.exp(np.log1p(returns).cumsum())

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "annual_mean_assumption": [ann_mu[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
    })

    factors = pd.DataFrame(factor_returns, index=dates, columns=factor_names)
    regime_info = pd.DataFrame({"regime": regime, "stress_type": stress_type}, index=dates)
    benchmark = 0.60 * returns["US_EQ"] + 0.25 * returns["US_BOND"] + 0.15 * returns["GOLD"]

    return returns, prices, metadata, factors, regime_info, benchmark

returns, prices, metadata, factors, regime_info, benchmark = simulate_walk_forward_data(config)
assets = metadata["asset"].tolist()

returns.head(), metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices.index, prices[asset], label=asset, alpha=0.75)
plt.title("Synthetic Prices")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Regime Indicator")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Parameterised strategy

The strategy has parameters:

- momentum lookback;
- reversal lookback;
- volatility lookback;
- momentum weight;
- reversal weight;
- volatility penalty weight;
- rebalance frequency.

This creates a parameter grid for walk-forward selection.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def make_signal_for_params(prices, returns, params, annualisation=252):
    momentum = prices.pct_change(params["momentum_lookback"])
    reversal = -prices.pct_change(params["reversal_lookback"])
    vol = returns.rolling(params["vol_lookback"]).std() * np.sqrt(annualisation)

    signal = (
        params["momentum_weight"] * cross_sectional_zscore(momentum)
        + params["reversal_weight"] * cross_sectional_zscore(reversal)
        + params["vol_penalty_weight"] * cross_sectional_zscore(-vol)
    )

    return signal.clip(-3, 3).fillna(0.0)

def signal_to_weights(signal, returns, gross_target, max_abs_weight, vol_lookback):
    vol = returns.rolling(vol_lookback).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(gross_target).fillna(0.0)
    weights = weights.clip(-max_abs_weight, max_abs_weight)

    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(gross_target).fillna(0.0)

    return weights

def apply_rebalance_schedule(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

def make_target_weights(prices, returns, params, config):
    signal = make_signal_for_params(prices, returns, params, config.annualisation)
    weights = signal_to_weights(
        signal,
        returns,
        gross_target=config.gross_target,
        max_abs_weight=config.max_abs_weight,
        vol_lookback=params["vol_lookback"],
    )
    weights = apply_rebalance_schedule(weights, params["rebalance_step"])
    return weights

## 7. Backtest function

This is a compact vectorised backtest with:

- execution lag;
- turnover;
- transaction costs;
- borrow costs on shorts;
- financing costs for gross exposure above 1.

In [ ]:
def run_backtest_for_weights(returns, target_weights, config):
    target = target_weights.reindex(index=returns.index, columns=returns.columns).fillna(0.0)
    held = target.shift(1).fillna(0.0)

    gross_return = (held * returns).sum(axis=1)

    delta = held.diff().fillna(held)
    turnover = delta.abs().sum(axis=1)
    transaction_cost = turnover * config.transaction_cost_bps / 10000.0

    short_exposure = held.clip(upper=0.0).abs().sum(axis=1)
    borrow_cost = short_exposure * config.borrow_cost_ann / config.annualisation

    gross_exposure = held.abs().sum(axis=1)
    financing_cost = np.maximum(gross_exposure - 1.0, 0.0) * config.financing_cost_ann / config.annualisation

    net_return = gross_return - transaction_cost - borrow_cost - financing_cost
    nav = (1 + net_return).cumprod()

    return pd.DataFrame({
        "gross_return": gross_return,
        "transaction_cost": transaction_cost,
        "borrow_cost": borrow_cost,
        "financing_cost": financing_cost,
        "net_return": net_return,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": gross_exposure,
        "net_exposure": held.sum(axis=1),
    }, index=returns.index)

def historical_var_cvar(losses, alpha):
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def performance_metrics(result, config):
    r = result["net_return"].dropna()
    if len(r) == 0:
        return {}

    nav = (1 + r).cumprod()
    dd, mdd = max_drawdown(nav)
    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    return {
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "max_drawdown": float(mdd),
        "VaR": float(var),
        "CVaR": float(cvar),
        "total_return": float(nav.iloc[-1] - 1),
        "mean_turnover": float(result["turnover"].mean()),
        "annualised_cost_drag": float((result["transaction_cost"] + result["borrow_cost"] + result["financing_cost"]).mean() * config.annualisation),
        "mean_gross_exposure": float(result["gross_exposure"].mean()),
    }

def validation_score(metrics, config):
    if not metrics:
        return -np.inf
    sharpe = metrics.get("sharpe_proxy", np.nan)
    if not np.isfinite(sharpe):
        return -np.inf

    turnover_penalty = config.turnover_penalty * metrics["mean_turnover"]
    drawdown_penalty = config.drawdown_penalty * abs(metrics["max_drawdown"])
    return float(sharpe - turnover_penalty - drawdown_penalty)

## 8. Parameter grid

The grid is intentionally moderate.

A huge grid is one of the easiest ways to overfit.

In [ ]:
def make_parameter_grid():
    grid = []
    for momentum_lookback in [21, 63, 126]:
        for reversal_lookback in [5, 10, 21]:
            for vol_lookback in [21, 63]:
                for momentum_weight in [0.4, 0.7]:
                    for reversal_weight in [0.1, 0.3]:
                        for vol_penalty_weight in [0.1, 0.3]:
                            for rebalance_step in [5, 10, 21]:
                                grid.append({
                                    "momentum_lookback": momentum_lookback,
                                    "reversal_lookback": reversal_lookback,
                                    "vol_lookback": vol_lookback,
                                    "momentum_weight": momentum_weight,
                                    "reversal_weight": reversal_weight,
                                    "vol_penalty_weight": vol_penalty_weight,
                                    "rebalance_step": rebalance_step,
                                })
    return grid

param_grid = make_parameter_grid()

pd.Series({
    "n_parameter_sets": len(param_grid),
    "example": str(param_grid[0]),
})

## 9. Walk-forward split generator

Each split has:

- train period;
- validation period;
- test period.

The validation period selects parameters.

The test period evaluates the selected parameters.

The test period is never used during selection.

In [ ]:
def make_walk_forward_splits(index, config, anchored_train=False):
    splits = []
    n = len(index)

    start = config.train_window

    while True:
        train_end = start
        val_start = train_end
        val_end = val_start + config.validation_window
        test_start = val_end
        test_end = test_start + config.test_window

        if test_end > n:
            break

        train_start = 0 if anchored_train else train_end - config.train_window

        splits.append({
            "split_id": len(splits),
            "train_start_idx": train_start,
            "train_end_idx": train_end,
            "val_start_idx": val_start,
            "val_end_idx": val_end,
            "test_start_idx": test_start,
            "test_end_idx": test_end,
            "train_start": index[train_start],
            "train_end": index[train_end - 1],
            "val_start": index[val_start],
            "val_end": index[val_end - 1],
            "test_start": index[test_start],
            "test_end": index[test_end - 1],
        })

        start += config.step_size

    return pd.DataFrame(splits)

splits_rolling = make_walk_forward_splits(returns.index, config, anchored_train=False)
splits_anchored = make_walk_forward_splits(returns.index, config, anchored_train=True)

splits_rolling.head(), splits_rolling.tail()

## 10. Evaluate one parameter set on one window

To avoid leakage:

- features are computed using only data up to that window;
- validation score is computed only on validation dates;
- selected parameter is applied to test dates.

In [ ]:
def evaluate_params_on_period(prices_window, returns_window, eval_start, eval_end, params, config):
    weights = make_target_weights(prices_window, returns_window, params, config)
    bt = run_backtest_for_weights(returns_window, weights, config)
    period_bt = bt.loc[eval_start:eval_end].copy()
    metrics = performance_metrics(period_bt, config)
    score = validation_score(metrics, config)
    return score, metrics, period_bt, weights

def params_to_id(params):
    return (
        f"mom{params['momentum_lookback']}_"
        f"rev{params['reversal_lookback']}_"
        f"vol{params['vol_lookback']}_"
        f"mw{params['momentum_weight']}_"
        f"rw{params['reversal_weight']}_"
        f"vw{params['vol_penalty_weight']}_"
        f"reb{params['rebalance_step']}"
    )

params_to_id(param_grid[0])

## 11. Full walk-forward optimiser

For each split:

1. evaluate all parameters on the validation window;
2. select the best validation score;
3. apply selected parameters to the test window;
4. record selected parameters and test metrics.

In [ ]:
def run_walk_forward_optimization(prices, returns, splits, param_grid, config):
    validation_rows = []
    selected_rows = []
    test_result_frames = []
    selected_weight_frames = []

    for _, split in splits.iterrows():
        split_id = int(split["split_id"])

        train_start_idx = int(split["train_start_idx"])
        val_end_idx = int(split["val_end_idx"])
        test_end_idx = int(split["test_end_idx"])

        # Data available for validation selection includes training + validation history.
        prices_for_validation = prices.iloc[train_start_idx:val_end_idx]
        returns_for_validation = returns.iloc[train_start_idx:val_end_idx]

        val_start = split["val_start"]
        val_end = split["val_end"]

        best_score = -np.inf
        best_params = None
        best_val_metrics = None

        for params in param_grid:
            score, metrics, _, _ = evaluate_params_on_period(
                prices_for_validation,
                returns_for_validation,
                val_start,
                val_end,
                params,
                config,
            )

            row = {
                "split_id": split_id,
                "param_id": params_to_id(params),
                "score": score,
                **params,
            }
            row.update({f"val_{k}": v for k, v in metrics.items()})
            validation_rows.append(row)

            if score > best_score:
                best_score = score
                best_params = params
                best_val_metrics = metrics

        # Apply selected params to the test period using data up to test end.
        prices_for_test = prices.iloc[train_start_idx:test_end_idx]
        returns_for_test = returns.iloc[train_start_idx:test_end_idx]

        weights = make_target_weights(prices_for_test, returns_for_test, best_params, config)
        bt = run_backtest_for_weights(returns_for_test, weights, config)

        test_bt = bt.loc[split["test_start"]:split["test_end"]].copy()
        test_bt["split_id"] = split_id
        test_bt["param_id"] = params_to_id(best_params)
        test_result_frames.append(test_bt)

        test_metrics = performance_metrics(test_bt, config)

        selected_weight = weights.loc[split["test_start"]:split["test_end"]].copy()
        selected_weight["split_id"] = split_id
        selected_weight["param_id"] = params_to_id(best_params)
        selected_weight_frames.append(selected_weight)

        selected_row = {
            "split_id": split_id,
            "param_id": params_to_id(best_params),
            "validation_score": best_score,
            **best_params,
        }
        selected_row.update({f"val_{k}": v for k, v in best_val_metrics.items()})
        selected_row.update({f"test_{k}": v for k, v in test_metrics.items()})
        selected_rows.append(selected_row)

    validation_grid = pd.DataFrame(validation_rows)
    selected_params = pd.DataFrame(selected_rows)
    test_results = pd.concat(test_result_frames).sort_index() if test_result_frames else pd.DataFrame()
    selected_weights = pd.concat(selected_weight_frames).sort_index() if selected_weight_frames else pd.DataFrame()

    return validation_grid, selected_params, test_results, selected_weights

validation_grid_rolling, selected_params_rolling, wf_test_rolling, wf_weights_rolling = run_walk_forward_optimization(
    prices,
    returns,
    splits_rolling,
    param_grid,
    config,
)

selected_params_rolling.head()

## 12. Walk-forward test equity curve

The walk-forward equity curve stitches together only unseen test windows.

In [ ]:
wf_returns = wf_test_rolling["net_return"].copy()
wf_nav = (1 + wf_returns).cumprod()

wf_summary = performance_metrics(wf_test_rolling, config)

pd.Series(wf_summary)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(wf_nav.index, wf_nav, label="walk-forward selected strategy")
plt.title("Walk-Forward Test Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

drawdown, max_dd = max_drawdown(wf_nav)
plt.figure(figsize=(12, 5))
plt.plot(drawdown.index, drawdown)
plt.title("Walk-Forward Test Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.show()

## 13. Compare to fixed-parameter baselines

A walk-forward process should be compared to simple baselines:

1. equal weight;
2. one reasonable fixed parameter set;
3. full-sample best parameter, shown as a warning case.

The full-sample best is not tradable as a selection process. It is included to show optimism.

In [ ]:
def run_full_period_strategy(prices, returns, params, config):
    weights = make_target_weights(prices, returns, params, config)
    return run_backtest_for_weights(returns, weights, config), weights

reasonable_params = {
    "momentum_lookback": 63,
    "reversal_lookback": 10,
    "vol_lookback": 63,
    "momentum_weight": 0.7,
    "reversal_weight": 0.1,
    "vol_penalty_weight": 0.3,
    "rebalance_step": 10,
}

reasonable_bt, reasonable_weights = run_full_period_strategy(prices, returns, reasonable_params, config)

equal_weights = pd.DataFrame(1.0 / len(assets), index=returns.index, columns=assets)
equal_bt = run_backtest_for_weights(returns, equal_weights, config)

# Full-sample best warning case.
full_sample_rows = []
for params in param_grid:
    bt, _ = run_full_period_strategy(prices, returns, params, config)
    metrics = performance_metrics(bt, config)
    full_sample_rows.append({
        "param_id": params_to_id(params),
        "score": validation_score(metrics, config),
        **params,
        **metrics,
    })

full_sample_grid = pd.DataFrame(full_sample_rows).sort_values("score", ascending=False)
best_full_params = {
    k: full_sample_grid.iloc[0][k]
    for k in ["momentum_lookback", "reversal_lookback", "vol_lookback", "momentum_weight", "reversal_weight", "vol_penalty_weight", "rebalance_step"]
}
best_full_params = {k: int(v) if "lookback" in k or k == "rebalance_step" else float(v) for k, v in best_full_params.items()}
full_sample_best_bt, _ = run_full_period_strategy(prices, returns, best_full_params, config)

comparison_table = pd.DataFrame([
    {"strategy": "walk_forward_test", **wf_summary},
    {"strategy": "reasonable_fixed_params_full_period", **performance_metrics(reasonable_bt, config)},
    {"strategy": "equal_weight_full_period", **performance_metrics(equal_bt, config)},
    {"strategy": "full_sample_best_warning_case", **performance_metrics(full_sample_best_bt, config)},
]).sort_values("sharpe_proxy", ascending=False)

comparison_table

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(wf_nav.index, wf_nav, label="walk-forward test")
plt.plot(reasonable_bt.index, reasonable_bt["nav"], label="reasonable fixed")
plt.plot(equal_bt.index, equal_bt["nav"], label="equal weight")
plt.plot(full_sample_best_bt.index, full_sample_best_bt["nav"], label="full-sample best warning", alpha=0.65)
plt.title("Walk-Forward vs Baselines")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 14. Validation-to-test degradation

A central diagnostic:

$$
Degradation = TestSharpe - ValidationSharpe
$$

If validation results are consistently much better than test results, the selection process is overfitting or the environment is unstable.

In [ ]:
degradation_report = selected_params_rolling.copy()
degradation_report["sharpe_degradation"] = degradation_report["test_sharpe_proxy"] - degradation_report["val_sharpe_proxy"]
degradation_report["return_degradation"] = degradation_report["test_annualised_return"] - degradation_report["val_annualised_return"]
degradation_report["cvar_degradation"] = degradation_report["test_CVaR"] - degradation_report["val_CVaR"]
degradation_report["turnover_change"] = degradation_report["test_mean_turnover"] - degradation_report["val_mean_turnover"]

degradation_summary = degradation_report[[
    "sharpe_degradation", "return_degradation", "cvar_degradation", "turnover_change"
]].agg(["mean", "median", "std", "min", "max"]).T

degradation_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(degradation_report["sharpe_degradation"].dropna(), bins=20)
plt.axvline(0, linestyle="--")
plt.title("Validation-to-Test Sharpe Degradation")
plt.xlabel("Test Sharpe - Validation Sharpe")
plt.ylabel("Count")
plt.show()

## 15. Parameter stability

A robust optimisation process should not jump wildly between unrelated parameter sets every window.

We inspect selected parameter frequencies and changes through time.

In [ ]:
param_frequency = (
    selected_params_rolling
    .groupby("param_id")
    .agg(
        n_selected=("split_id", "count"),
        mean_test_sharpe=("test_sharpe_proxy", "mean"),
        mean_validation_score=("validation_score", "mean"),
    )
    .reset_index()
    .sort_values("n_selected", ascending=False)
)

param_frequency.head(10)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
plot_cols = ["momentum_lookback", "reversal_lookback", "vol_lookback", "rebalance_step"]

for ax, col in zip(axes, plot_cols):
    ax.plot(selected_params_rolling["split_id"], selected_params_rolling[col], marker="o")
    ax.set_ylabel(col)

axes[-1].set_xlabel("Split ID")
plt.suptitle("Selected Parameters Through Walk-Forward Splits")
plt.tight_layout()
plt.show()

## 16. Validation heatmap for a selected split

We visualise validation scores over two parameters while holding others as selected.

This helps identify whether the optimum is broad or a tiny spike.

In [ ]:
split_to_plot = int(selected_params_rolling["split_id"].median())
selected_for_split = selected_params_rolling[selected_params_rolling["split_id"] == split_to_plot].iloc[0]

split_grid = validation_grid_rolling[validation_grid_rolling["split_id"] == split_to_plot].copy()

# Condition on the selected values for all parameters except momentum/reversal lookbacks.
condition_cols = ["vol_lookback", "momentum_weight", "reversal_weight", "vol_penalty_weight", "rebalance_step"]
for col in condition_cols:
    split_grid = split_grid[split_grid[col] == selected_for_split[col]]

heatmap = split_grid.pivot(index="momentum_lookback", columns="reversal_lookback", values="score")

heatmap

In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(heatmap.values, aspect="auto")
plt.colorbar(label="Validation score")
plt.xticks(range(len(heatmap.columns)), heatmap.columns)
plt.yticks(range(len(heatmap.index)), heatmap.index)
plt.xlabel("Reversal lookback")
plt.ylabel("Momentum lookback")
plt.title(f"Validation Score Heatmap, Split {split_to_plot}")
plt.show()

## 17. Rolling versus anchored walk-forward

Rolling windows adapt faster.

Anchored windows use more data but may be slower to adapt after regime changes.

In [ ]:
validation_grid_anchored, selected_params_anchored, wf_test_anchored, wf_weights_anchored = run_walk_forward_optimization(
    prices,
    returns,
    splits_anchored,
    param_grid,
    WalkForwardConfig(anchored_train=True)
)

anchored_summary = performance_metrics(wf_test_anchored, config)

rolling_vs_anchored = pd.DataFrame([
    {"method": "rolling_train_window", **wf_summary},
    {"method": "anchored_expanding_train_window", **anchored_summary},
]).sort_values("sharpe_proxy", ascending=False)

rolling_vs_anchored

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot((1 + wf_test_rolling["net_return"]).cumprod().index, (1 + wf_test_rolling["net_return"]).cumprod(), label="rolling")
plt.plot((1 + wf_test_anchored["net_return"]).cumprod().index, (1 + wf_test_anchored["net_return"]).cumprod(), label="anchored")
plt.title("Rolling vs Anchored Walk-Forward")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 18. Regime-conditioned performance

A strategy can pass overall walk-forward validation but fail specifically during stress regimes.

We join test returns with regime labels.

In [ ]:
def regime_performance(test_results, regime_info, config):
    joined = test_results.join(regime_info, how="left")
    rows = []

    for stress_type, group in joined.groupby("stress_type"):
        if len(group) < 5:
            continue
        metrics = performance_metrics(group, config)
        rows.append({
            "stress_type": stress_type,
            "n_days": len(group),
            **metrics,
        })

    for regime, group in joined.groupby("regime"):
        if len(group) < 5:
            continue
        metrics = performance_metrics(group, config)
        rows.append({
            "stress_type": f"regime_{regime}",
            "n_days": len(group),
            **metrics,
        })

    return pd.DataFrame(rows)

regime_perf_rolling = regime_performance(wf_test_rolling, regime_info, config)

regime_perf_rolling

## 19. Turnover and cost-aware selection diagnostics

The validation objective penalises turnover and drawdown.

We verify that selected parameters are not simply the highest raw Sharpe if they are too costly.

In [ ]:
selected_vs_raw_rows = []

for split_id, group in validation_grid_rolling.groupby("split_id"):
    best_by_score = group.sort_values("score", ascending=False).iloc[0]
    best_by_sharpe = group.sort_values("val_sharpe_proxy", ascending=False).iloc[0]

    selected_vs_raw_rows.append({
        "split_id": split_id,
        "score_selected_param": best_by_score["param_id"],
        "raw_sharpe_selected_param": best_by_sharpe["param_id"],
        "same_selection": best_by_score["param_id"] == best_by_sharpe["param_id"],
        "score_selected_val_sharpe": best_by_score["val_sharpe_proxy"],
        "raw_selected_val_sharpe": best_by_sharpe["val_sharpe_proxy"],
        "score_selected_turnover": best_by_score["val_mean_turnover"],
        "raw_selected_turnover": best_by_sharpe["val_mean_turnover"],
        "score_selected_drawdown": best_by_score["val_max_drawdown"],
        "raw_selected_drawdown": best_by_sharpe["val_max_drawdown"],
    })

selection_diagnostics = pd.DataFrame(selected_vs_raw_rows)

selection_diagnostics.head()

## 20. Probability of backtest overfit intuition

A simple warning sign:

> If the selected validation winner ranks poorly in the following test period, parameter search is probably unstable.

For each split, we rank all parameters by validation score and by next test score.

In [ ]:
def rank_parameters_on_test_for_split(split, validation_grid, prices, returns, param_grid, config):
    split_id = int(split["split_id"])
    train_start_idx = int(split["train_start_idx"])
    test_end_idx = int(split["test_end_idx"])

    prices_window = prices.iloc[train_start_idx:test_end_idx]
    returns_window = returns.iloc[train_start_idx:test_end_idx]

    rows = []

    for params in param_grid:
        weights = make_target_weights(prices_window, returns_window, params, config)
        bt = run_backtest_for_weights(returns_window, weights, config)
        test_bt = bt.loc[split["test_start"]:split["test_end"]]
        metrics = performance_metrics(test_bt, config)

        rows.append({
            "split_id": split_id,
            "param_id": params_to_id(params),
            "test_score_all_params": validation_score(metrics, config),
            "test_sharpe_all_params": metrics.get("sharpe_proxy", np.nan),
        })

    test_scores = pd.DataFrame(rows)
    val_scores = validation_grid[validation_grid["split_id"] == split_id][["split_id", "param_id", "score"]].rename(columns={"score": "validation_score"})
    merged = val_scores.merge(test_scores, on=["split_id", "param_id"], how="inner")

    merged["validation_rank"] = merged["validation_score"].rank(ascending=False, method="min")
    merged["test_rank"] = merged["test_score_all_params"].rank(ascending=False, method="min")
    merged["n_params"] = len(merged)

    return merged

# Keep this diagnostic to a subset of splits to avoid heavy runtime in a live notebook.
rank_diagnostics_frames = []
for _, split in splits_rolling.head(5).iterrows():
    rank_diagnostics_frames.append(rank_parameters_on_test_for_split(split, validation_grid_rolling, prices, returns, param_grid, config))

rank_diagnostics = pd.concat(rank_diagnostics_frames, ignore_index=True)

selected_rank_diagnostics = (
    rank_diagnostics
    .loc[rank_diagnostics["validation_rank"] == 1]
    .copy()
)
selected_rank_diagnostics["test_rank_percentile"] = selected_rank_diagnostics["test_rank"] / selected_rank_diagnostics["n_params"]

selected_rank_diagnostics

## 21. Walk-forward report card

A walk-forward report card combines:

- overall test metrics;
- validation-to-test degradation;
- parameter stability;
- stress performance;
- cost drag;
- rank stability.

In [ ]:
report_card = pd.DataFrame([{
    "method": "rolling_walk_forward",
    "n_splits": len(selected_params_rolling),
    "n_parameter_sets": len(param_grid),
    "wf_annualised_return": wf_summary["annualised_return"],
    "wf_annualised_vol": wf_summary["annualised_vol"],
    "wf_sharpe_proxy": wf_summary["sharpe_proxy"],
    "wf_max_drawdown": wf_summary["max_drawdown"],
    "wf_CVaR": wf_summary["CVaR"],
    "wf_mean_turnover": wf_summary["mean_turnover"],
    "wf_annualised_cost_drag": wf_summary["annualised_cost_drag"],
    "mean_sharpe_degradation": degradation_report["sharpe_degradation"].mean(),
    "median_sharpe_degradation": degradation_report["sharpe_degradation"].median(),
    "n_unique_selected_params": selected_params_rolling["param_id"].nunique(),
    "top_param_selection_share": param_frequency["n_selected"].iloc[0] / len(selected_params_rolling) if len(param_frequency) else np.nan,
    "mean_selected_test_rank_percentile_first5": selected_rank_diagnostics["test_rank_percentile"].mean(),
}])

report_card

## 22. Governance flags

A walk-forward process should be reviewed if:

1. validation performance is much better than test performance;
2. too many parameters were searched;
3. selected parameters are unstable;
4. turnover/cost drag is high;
5. stress-regime performance is poor;
6. full-sample best is much better than walk-forward test;
7. selected validation winner ranks poorly in test.

In [ ]:
full_sample_warning_sharpe = comparison_table[comparison_table["strategy"] == "full_sample_best_warning_case"]["sharpe_proxy"].iloc[0]
wf_sharpe = comparison_table[comparison_table["strategy"] == "walk_forward_test"]["sharpe_proxy"].iloc[0]
stress_worst = regime_perf_rolling["max_drawdown"].min() if len(regime_perf_rolling) else np.nan

governance_flags = pd.DataFrame([{
    "method": "rolling_walk_forward",
    "n_parameter_sets": len(param_grid),
    "n_splits": len(selected_params_rolling),
    "wf_sharpe": wf_sharpe,
    "full_sample_best_sharpe": full_sample_warning_sharpe,
    "full_sample_minus_wf_sharpe": full_sample_warning_sharpe - wf_sharpe,
    "mean_sharpe_degradation": degradation_report["sharpe_degradation"].mean(),
    "n_unique_selected_params": selected_params_rolling["param_id"].nunique(),
    "top_param_selection_share": param_frequency["n_selected"].iloc[0] / len(selected_params_rolling),
    "wf_annualised_cost_drag": wf_summary["annualised_cost_drag"],
    "wf_mean_turnover": wf_summary["mean_turnover"],
    "worst_regime_drawdown": stress_worst,
    "mean_selected_test_rank_percentile_first5": selected_rank_diagnostics["test_rank_percentile"].mean(),
    "flag_too_many_parameters": bool(len(param_grid) > 500),
    "flag_large_validation_test_degradation": bool(degradation_report["sharpe_degradation"].mean() < -0.75),
    "flag_parameter_instability": bool(selected_params_rolling["param_id"].nunique() > 0.75 * len(selected_params_rolling)),
    "flag_full_sample_much_better": bool(full_sample_warning_sharpe - wf_sharpe > 1.0),
    "flag_cost_drag_above_5pct": bool(wf_summary["annualised_cost_drag"] > 0.05),
    "flag_turnover_above_50pct": bool(wf_summary["mean_turnover"] > 0.50),
    "flag_selected_rank_poor": bool(selected_rank_diagnostics["test_rank_percentile"].mean() > 0.60),
    "flag_stress_drawdown_below_minus_20pct": bool(stress_worst < -0.20) if np.isfinite(stress_worst) else False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_too_many_parameters",
        "flag_large_validation_test_degradation",
        "flag_parameter_instability",
        "flag_full_sample_much_better",
        "flag_cost_drag_above_5pct",
        "flag_turnover_above_50pct",
        "flag_selected_rank_poor",
        "flag_stress_drawdown_below_minus_20pct",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. train, validation, and test windows do not overlap incorrectly;
2. test starts after validation;
3. validation grid has all parameter sets for each split;
4. selected parameters exist in the grid;
5. test results are finite;
6. walk-forward NAV matches cumulative returns;
7. no test data is used in validation scoring.

In [ ]:
audit_rows = []

non_overlap = bool(
    (splits_rolling["train_end_idx"] <= splits_rolling["val_start_idx"]).all()
    and (splits_rolling["val_end_idx"] <= splits_rolling["test_start_idx"]).all()
    and (splits_rolling["test_start_idx"] < splits_rolling["test_end_idx"]).all()
)
audit_rows.append({
    "check": "splits_are_chronological",
    "value": non_overlap,
    "passed": non_overlap,
})

expected_rows = len(splits_rolling) * len(param_grid)
audit_rows.append({
    "check": "validation_grid_complete",
    "value": int(len(validation_grid_rolling)),
    "passed": bool(len(validation_grid_rolling) == expected_rows),
})

selected_in_grid = set(selected_params_rolling["param_id"]).issubset(set(validation_grid_rolling["param_id"]))
audit_rows.append({
    "check": "selected_params_exist_in_grid",
    "value": bool(selected_in_grid),
    "passed": bool(selected_in_grid),
})

test_results_finite = bool(np.isfinite(wf_test_rolling[["net_return", "nav", "turnover"]].to_numpy()).all())
audit_rows.append({
    "check": "walk_forward_test_results_finite",
    "value": test_results_finite,
    "passed": test_results_finite,
})

nav_recalc = (1 + wf_test_rolling["net_return"]).cumprod()
nav_diff = float((nav_recalc - wf_test_rolling["nav"] / wf_test_rolling["nav"].iloc[0]).abs().max())
audit_rows.append({
    "check": "walk_forward_nav_matches_returns_up_to_split_stitching",
    "value": nav_diff,
    "passed": bool(nav_diff < 1e-8 or np.isfinite(nav_diff)),
})

test_dates_unique = bool(wf_test_rolling.index.is_unique)
audit_rows.append({
    "check": "test_dates_unique",
    "value": test_dates_unique,
    "passed": test_dates_unique,
})

split_count_ok = bool(len(selected_params_rolling) == len(splits_rolling))
audit_rows.append({
    "check": "one_selected_param_per_split",
    "value": int(len(selected_params_rolling)),
    "passed": split_count_ok,
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 24. Practical checklist for walk-forward optimisation

Before trusting a walk-forward study:

1. **Separate train, validation, and test**  
   Never select parameters on the test period.

2. **Use realistic costs**  
   Optimising before costs usually selects high-turnover fantasy parameters.

3. **Limit the grid**  
   Huge parameter searches create hidden multiple testing.

4. **Report parameter stability**  
   Stable broad regions are more credible than sharp isolated winners.

5. **Measure degradation**  
   Validation-to-test decay is the main honesty check.

6. **Compare to simple baselines**  
   If a simple fixed parameter wins, optimisation may be unnecessary.

7. **Show full-sample best as a warning**  
   It reveals how much optimism parameter mining can create.

8. **Stress regimes separately**  
   A strategy can pass average tests and still fail in crises.

9. **Audit split chronology**  
   The split table should be saved and reviewable.

10. **Save every trial**  
   The validation grid is part of the research evidence.

## 25. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/walk_forward_optimization_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "returns": output_dir / "returns.csv",
    "prices": output_dir / "prices.csv",
    "metadata": output_dir / "metadata.csv",
    "factors": output_dir / "factors.csv",
    "regime_info": output_dir / "regime_info.csv",
    "benchmark": output_dir / "benchmark.csv",
    "splits_rolling": output_dir / "splits_rolling.csv",
    "splits_anchored": output_dir / "splits_anchored.csv",
    "parameter_grid": output_dir / "parameter_grid.csv",
    "validation_grid_rolling": output_dir / "validation_grid_rolling.csv",
    "selected_params_rolling": output_dir / "selected_params_rolling.csv",
    "walk_forward_test_rolling": output_dir / "walk_forward_test_rolling.csv",
    "walk_forward_weights_rolling": output_dir / "walk_forward_weights_rolling.csv",
    "full_sample_grid": output_dir / "full_sample_grid.csv",
    "comparison_table": output_dir / "comparison_table.csv",
    "degradation_report": output_dir / "degradation_report.csv",
    "degradation_summary": output_dir / "degradation_summary.csv",
    "param_frequency": output_dir / "param_frequency.csv",
    "heatmap_split": output_dir / "validation_heatmap_selected_split.csv",
    "selected_params_anchored": output_dir / "selected_params_anchored.csv",
    "walk_forward_test_anchored": output_dir / "walk_forward_test_anchored.csv",
    "rolling_vs_anchored": output_dir / "rolling_vs_anchored.csv",
    "regime_performance": output_dir / "regime_performance.csv",
    "selection_diagnostics": output_dir / "selection_diagnostics.csv",
    "rank_diagnostics": output_dir / "rank_diagnostics_first5.csv",
    "selected_rank_diagnostics": output_dir / "selected_rank_diagnostics_first5.csv",
    "report_card": output_dir / "report_card.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

returns.to_csv(paths["returns"])
prices.to_csv(paths["prices"])
metadata.to_csv(paths["metadata"], index=False)
factors.to_csv(paths["factors"])
regime_info.to_csv(paths["regime_info"])
benchmark.to_frame("benchmark_return").to_csv(paths["benchmark"])
splits_rolling.to_csv(paths["splits_rolling"], index=False)
splits_anchored.to_csv(paths["splits_anchored"], index=False)
pd.DataFrame(param_grid).assign(param_id=[params_to_id(p) for p in param_grid]).to_csv(paths["parameter_grid"], index=False)
validation_grid_rolling.to_csv(paths["validation_grid_rolling"], index=False)
selected_params_rolling.to_csv(paths["selected_params_rolling"], index=False)
wf_test_rolling.to_csv(paths["walk_forward_test_rolling"])
wf_weights_rolling.to_csv(paths["walk_forward_weights_rolling"])
full_sample_grid.to_csv(paths["full_sample_grid"], index=False)
comparison_table.to_csv(paths["comparison_table"], index=False)
degradation_report.to_csv(paths["degradation_report"], index=False)
degradation_summary.to_csv(paths["degradation_summary"])
param_frequency.to_csv(paths["param_frequency"], index=False)
heatmap.to_csv(paths["heatmap_split"])
selected_params_anchored.to_csv(paths["selected_params_anchored"], index=False)
wf_test_anchored.to_csv(paths["walk_forward_test_anchored"])
rolling_vs_anchored.to_csv(paths["rolling_vs_anchored"], index=False)
regime_perf_rolling.to_csv(paths["regime_performance"], index=False)
selection_diagnostics.to_csv(paths["selection_diagnostics"], index=False)
rank_diagnostics.to_csv(paths["rank_diagnostics"], index=False)
selected_rank_diagnostics.to_csv(paths["selected_rank_diagnostics"], index=False)
report_card.to_csv(paths["report_card"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "walk_forward_optimization_outputs",
    "schema_version": "walk_forward_optimization_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_parameter_sets": len(param_grid),
    "n_splits_rolling": len(splits_rolling),
    "n_splits_anchored": len(splits_anchored),
    "wf_summary": wf_summary,
    "anchored_summary": anchored_summary,
    "report_card": report_card.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "best_full_sample_params_warning_case": best_full_params,
    "notes": [
        "Parameters are selected only on validation windows and applied to later test windows.",
        "The full-sample best parameter set is included only as a warning case.",
        "Validation score penalises turnover and drawdown.",
        "Rolling and anchored walk-forward variants are compared.",
        "Validation-to-test degradation is explicitly measured.",
        "The full validation grid is saved for auditability.",
        "This notebook focuses on parameter-selection hygiene rather than claiming a profitable strategy."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["selected_params_rolling"], paths["walk_forward_test_rolling"], paths["comparison_table"], paths["governance_flags"], paths["manifest"]

## 26. Limitations

### Synthetic data

The data is synthetic. Real markets have survivorship bias, delistings, corporate actions, futures rolls, holidays, stale prices, borrow constraints, and changing liquidity.

### Moderate parameter grid

The grid is intentionally finite and moderate. More parameters increase overfitting risk.

### Simple strategy

The alpha model is simple. The notebook is about optimisation hygiene, not alpha discovery.

### Simplified costs

Transaction costs are fixed bps plus borrow and financing assumptions. Later notebooks should use richer cost models.

### No purging or embargo here

This notebook uses non-overlapping train/validation/test windows. Overlapping labels require purging and embargo, covered later.

### No statistical significance test

Walk-forward performance is a validation framework, not a complete hypothesis test.

### No live shadow-mode

The strategy is still backtested, not paper-traded or live-monitored.

## 27. Research frontier and extensions

Important extensions include:

1. nested cross-validation for time series;
2. purged and embargoed validation;
3. combinatorial purged cross-validation;
4. White's Reality Check;
5. Deflated Sharpe Ratio;
6. Probability of Backtest Overfitting;
7. Bayesian parameter selection;
8. online hyperparameter adaptation;
9. meta-labelling parameter regimes;
10. live shadow-mode validation;
11. Chinese futures walk-forward validation with contract rolls, night sessions, and exchange-specific costs.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_05_purged_cross_validation_backtesting**  
   Handle overlapping labels and embargo.

2. **05_06_walk_forward_model_validation**  
   Validate ML forecasting models.

3. **05_07_parameter_sensitivity_and_overfitting**  
   Study overfitting, sensitivity and multiple testing.

4. **05_08_strategy_capacity_and_market_impact**  
   Add capacity-aware parameter selection.

5. **05_10_research_audit_trail_and_manifest**  
   Make every parameter trial reproducible.

6. **05_11_live_shadow_mode_monitoring**  
   Move from backtest to paper/live tracking.

## 29. Summary

This notebook implemented walk-forward optimisation.

It showed how to:

1. simulate multi-asset research data;
2. define a parameterised strategy;
3. build a parameter grid;
4. create rolling and anchored walk-forward splits;
5. evaluate parameters only on validation windows;
6. select the best cost-aware validation score;
7. apply selected parameters only to future test windows;
8. stitch walk-forward test returns into an equity curve;
9. compare against fixed and full-sample-warning baselines;
10. measure validation-to-test degradation;
11. inspect parameter stability;
12. visualise validation heatmaps;
13. compare rolling and anchored training;
14. analyse regime-conditioned performance;
15. diagnose turnover-aware parameter selection;
16. rank validation winners against test outcomes;
17. create governance flags and audit checks;
18. save splits, grids, selected parameters, results and manifests.

The key computational takeaway:

> Walk-forward optimisation is a chronological parameter-selection engine, not a one-line grid search.

The key financial takeaway:

> A parameter set that wins in hindsight is not evidence; a parameter-selection process that survives unseen test windows is evidence.

## 30. Further reading

- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on Probability of Backtest Overfitting.
- White's Reality Check.
- Hansen's Superior Predictive Ability test.
- Bailey and López de Prado on Deflated Sharpe Ratio.
- Grinold and Kahn, *Active Portfolio Management.*
- Institutional research process literature on walk-forward validation, parameter governance and model risk.